In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../../python')
from ddms.numeric import HandlerDamask

In [ ]:
%matplotlib inline

In [ ]:
yaml = '../../config/AA6111.yaml'
taskname = 's1a-g1-no_gnd'
target = '7min_298K'
rootdir = ''
solver = 'damask'
handler = HandlerDamask(yaml, taskname, target, rootdir)

# exp

## stress strain curve

In [ ]:
target = '7min'
handler = HandlerDamask(yaml, taskname, target, rootdir, solver)

_, ax1 = plt.subplots(figsize=(6,4), tight_layout=True)
for _ in handler.params_updater():
	if handler.prm.cond=='7min' and handler.prm.T==298:
		continue
	exp_strain, exp_stress, exp_YSx, exp_YSy = handler.getEXP(plastic=False, UTS=False)
	# exp_strain, exp_stress = handler.smoothing(exp_strain, 1), handler.smoothing(exp_stress, 1)
	# ax1.plot(exp_strain, exp_stress, '-', alpha=0.8, label=handler.prm.cond)
	ax1.plot(exp_strain, exp_stress, '-', alpha=0.8, label=f'{handler.prm.T}K')

ax1.grid(alpha=0.2)
ax1.set_xlabel('strain')
ax1.set_ylabel('stress[MPa]')
# ax1.set_ylim(0, 80)
ax1.legend()
plt.show()

In [ ]:
target = '7min'
handler = HandlerDamask(yaml, taskname, target, rootdir, solver)

_, ax1 = plt.subplots(figsize=(6,4), tight_layout=True)
for _ in handler.params_updater():
	if handler.prm.cond=='7min' and handler.prm.T==298:
		continue
	exp_strain, exp_stress, exp_YSx, exp_YSy = handler.getEXP(plastic=True, UTS=False)
	# exp_strain, exp_stress = handler.smoothing(exp_strain, 3), handler.smoothing(exp_stress, 3)
	# ax1.plot(exp_strain, exp_stress, '-', alpha=0.8, label=handler.prm.cond)
	ax1.plot(exp_strain, exp_stress, '-', alpha=0.8, label=f'{handler.prm.T}K')

ax1.grid(alpha=0.2)
ax1.set_xlabel('plastic strain $\\varepsilon - \\varepsilon_y$')
ax1.set_ylabel('stress $\sigma - \sigma_y$[MPa]')
ax1.set_ylim(0, 80)
# ax1.set_ylim(-20, 80)
ax1.legend()
plt.show()

## YS hist

In [ ]:
target = '7min'
handler = HandlerDamask(yaml, taskname, target, rootdir, solver)

_, ax1 = plt.subplots(figsize=(6,4), tight_layout=True)
YSs = []
for _ in handler.params_updater():
	if handler.prm.cond=='7min' and handler.prm.T==298:
		continue
	exp_strain, exp_stress, exp_YSx, exp_YSy = handler.getEXP(plastic=False, UTS=False)
	# exp_strain, exp_stress = handler.smoothing(exp_strain, 10), handler.smoothing(exp_stress, 10)
	# ax1.plot(exp_strain, exp_stress, 'o', alpha=0.8, markersize=2, label=handler.prm.cond)
	YSs.append(exp_YSy[0])

_, bins, patches = ax1.hist(np.linspace(1,3,3), weights=YSs, bins=len(YSs), edgecolor='k', width=0.3)
for p, c in zip(patches, [u'#1f77b4', u'#ff7f0e', u'#2ca02c', u'#d62728']):
	p.set_facecolor(c)

ticks = [(patch.get_x() + (patch.get_x() + patch.get_width()))/2 for patch in patches]
# ticklabels = ['7min', '30min', '6hr', '168hr']
ticklabels = ['423K', '473K', '523K']

ax1.grid(alpha=0.2)
ax1.set_ylabel('yield stress[MPa]')
ax1.bar_label(patches)
ax1.set_xticks(ticks, ticklabels)
ax1.set(xlim=[0.5, 3.1], ylim=[0, 350])
plt.show()

## YS plot

In [ ]:
# exp + YSmodel
target = '298K'
handler = HandlerDamask(yaml, taskname, target, rootdir, solver)
ts = np.array([7*60, 30*60, 360*60, 168*60*60])
ss, pp, YS, expYS = [], [], [], []
for _ in handler.params_updater():
    handler.c_params()
    ss.append(sum(handler.prm.ss)/1e6)
    pp.append(handler.prm.pp/1e6)
    YS.append(handler.prm.YS/1e6)
    expYS.append(handler.prm.YS_exp)

# kinetics + YSmodel
rate = 0.001
strain = 168*60*60*rate 		# 168hr
numerical_inc = 1e-10 		# to prevent inf/nan
delta_t = 1e-1
times = np.arange(0, strain/rate+delta_t, delta_t)
ss_history = np.load("", allow_pickle=True)/1e6
pp_history = np.load("", allow_pickle=True)/1e6
YS_history = np.load("", allow_pickle=True)/1e6


fig, axes = plt.subplots(2, 1, figsize=(6, 3*2), tight_layout=True, gridspec_kw = {'hspace':0})
l1 = axes[0].plot(times, ss_history, '-', c='darkblue', label='solute')
axes[0].plot(ts, ss, '--o', c='darkblue')
axes[0].set(xscale='log', xlim=[1e0, 1e6], ylim=[0, 300],
            xticklabels=[], ylabel='$\sigma_s$[Mpa]')
axes[0].grid(alpha=0.2)

ax_ = axes[0].twinx()
l2 = ax_.plot(times, pp_history, '-', c='darkred', label='precipitate')
ax_.plot(ts, pp, '--o', c='darkred')
ax_.set(ylim=[0, 300], ylabel='$\sigma_p$[Mpa]')
lns = l1 + l2
labs = [l.get_label() for l in lns]
ax_.legend(lns, labs, loc='upper left')

axes[1].plot(times, YS_history, '-', c='k')
axes[1].plot(ts, YS, '--o', c='k')
axes[1].scatter(ts, expYS, c='r')
axes[1].set(xscale='log', xlim=[1e0, 1e6], ylim=[50, 350], yticks=np.linspace(50, 300, 6), 
            xlabel='time[s]', ylabel='$\sigma_y$[MPa]')
axes[1].grid(alpha=0.2)

# _l3 = axes[1].plot([], [], '-k', label='kinetics+strength model')
# _l4 = axes[1].plot([], [], '--ok', label='exp+strength model')
# _l5 = axes[1].scatter([], [], c='k', label='exp')
# # labels = ['kinetics+strength model','exp+strength model','exp']
# fig.legend([_l3[0].get_label(), _l4[0].get_label(), _l5.get_label()], 
#            loc='lower center', bbox_to_anchor=(0.5,-0.05), 
#            ncol=3, alignment='center', frameon=True)

for ax in [ax_, *axes]:
    ax.tick_params(which='major', axis='x', direction='in')
    ax.tick_params(which='major', axis='y', direction='in')
    ax.tick_params(which='minor', axis='x', direction='in')
    ax.tick_params(which='minor', axis='y', direction='in')

for ax in axes:
    for t in ts:
        ax.plot([t, t], [0, 1e25], '--k', alpha=0.2)

plt.show()

## hardrate

In [ ]:
target = '298K'
handler = HandlerDamask(yaml, taskname, target, rootdir, solver)

smt_stress = 100
smt_hard = 10

_, ax = plt.subplots(figsize=(6,4), tight_layout=True)
for _ in handler.params_updater():
	exp_strain, exp_stress, exp_YSx, exp_YSy = handler.getEXP(plastic=False, UTS=True)
	exp_strain, exp_stress = handler.smoothing(exp_strain, smt_stress), handler.smoothing(exp_stress, smt_stress)
	exp_hardrate = np.gradient(exp_stress, exp_strain)

	smt_exp_hardrate = handler.smoothing(exp_hardrate, smt_hard)
	line = ax.plot(exp_stress-exp_YSy, exp_hardrate, alpha=0.2)
	ax.plot(exp_stress[smt_hard-1:]-exp_YSy, smt_exp_hardrate, color=line[0].get_color(), label=handler.prm.cond)

ax.grid(alpha=0.2)
ax.set_xlabel('stress $\sigma - \sigma_y$[MPa]')
ax.set_ylabel('$\\theta$[MPa]')
ax.set_xlim(0, 100)
ax.set_ylim(0, 3000)
ax.legend()
plt.show()

## hardening behavior

In [ ]:
smt_stress = 100
smt_hard = 10
dtds_start = [18, 16, 30, 18]     # MPa

_, ax = plt.subplots(figsize=(6,4), tight_layout=True)
for i, _ in enumerate(handler.params_updater()):
	exp_strain, exp_stress, exp_YSx, exp_YSy = handler.getEXP(plastic=False, UTS=True)
	exp_strain, exp_stress = handler.smoothing(exp_strain, smt_stress), handler.smoothing(exp_stress, smt_stress)
	exp_hardrate = np.gradient(exp_stress, exp_strain)
	smt_exp_hardrate = handler.smoothing(exp_hardrate, smt_hard)

	wh_stress = exp_stress[smt_hard-1:]-exp_YSy
	dtds_mask = wh_stress>=dtds_start[i]
	dtds = np.gradient(smt_exp_hardrate[dtds_mask][::smt_hard], wh_stress[dtds_mask][::smt_hard])
	ax.scatter(exp_YSy, -dtds.mean(), label=handler.prm.cond)

	
ax.grid(alpha=0.2)
ax.set_xlabel('$\sigma_y$[MPa]')
ax.set_ylabel('$-\\frac{d\\theta}{d\sigma}$')
# ax.set_xlim(0, 100)
# ax.set_ylim(0, 3000)
ax.legend()
plt.show()

# simulation

## YS hist

In [ ]:
target = '298K'
handler = HandlerDamask(yaml, taskname, target, rootdir, solver)

_, ax1 = plt.subplots(figsize=(6,4), tight_layout=True)
exp_YSs = []
sim_YSs = []; sim_ss = []; sim_pp = []
width = 0.4
colors = [u'#1f77b4', u'#ff7f0e', u'#2ca02c', u'#d62728']
for _ in handler.params_updater():
	exp_strain, exp_stress, exp_YSx, exp_YSy = handler.getEXP(plastic=False, UTS=False)
	exp_YSs.append(exp_YSy[0])

	handler.c_params()
	sim_YSs.append(handler.prm.YS/1e6)
	sim_ss.append(sum(handler.prm.ss/1e6))
	sim_pp.append(handler.prm.pp/1e6)

bar_exp = ax1.bar(np.linspace(1,4,4), exp_YSs, width=width, edgecolor='k')
bar_sim = ax1.bar(np.linspace(1,4,4)+width, sim_YSs, width=width)
for b_exp, c in zip(bar_exp, colors):
	b_exp.set_facecolor(c)
ticks = [b.get_x()+b.get_width() for b in bar_exp]
ticklabels = ['7min', '30min', '6hr', '168hr']

for i, x in enumerate(np.linspace(1,4,4)+width):
	bar_i = ax1.bar(x, 10, bottom=0, width=width, edgecolor='k', facecolor=colors[i], hatch='///')
	bar_s = ax1.bar(x, sim_ss[i], bottom=bar_i[0].get_height(), width=width, edgecolor='k', facecolor=colors[i], hatch='o')
	bar_p = ax1.bar(x, sim_pp[i], bottom=bar_i[0].get_height()+bar_s[0].get_height(), width=width, edgecolor='k', facecolor=colors[i], hatch='/o')

ax1.bar_label(bar_exp, fmt='%.1f')
ax1.bar_label(bar_sim, fmt='%.1f')
ax1.set_ylabel('yield stress[MPa]')
ax1.set_xticks(ticks, ticklabels)
ax1.hist([], edgecolor='k', width=width, label='exp', hatch='', fill=False)
ax1.hist([], edgecolor='k', width=width, label='$\sigma_i$', hatch='///', fill=False)
ax1.hist([], edgecolor='k', width=width, label='$\sigma_s$', hatch='o', fill=False)
ax1.hist([], edgecolor='k', width=width, label='$\sigma_p$', hatch='/o', fill=False)
ax1.legend(handleheight=1.5, handlelength=2)

ax1.set(xlim=[0.25, 5.15], ylim=[0, 350])
ax1.grid(alpha=0.2)
plt.show()

## stress strain curve

In [ ]:
taskname = 's1ag1'
target = '30min'
rootdir = ''
handler = HandlerDamask(yaml, taskname, target, rootdir, solver)

# _, ax = plt.subplots(figsize=(6, 4), tight_layout=True)
for _ in handler.params_updater():
	handler.plotDAMASK3(plastic=True, UTS=False, markevery=50, smooth=True, reduced=True)

# strain = np.load("", allow_pickle=True)
# stress = np.load("", allow_pickle=True)
# ax.plot(strain, stress, '--r', label='approx., $f=1.50\%$')
# ax.legend(loc='upper left')

plt.show()

In [ ]:
sim_strain_stable = np.load("", allow_pickle=True)
sim_stress_stable = np.load("", allow_pickle=True)
sim_strain_nonstable = np.load("", allow_pickle=True)
sim_stress_nonstable = np.load("", allow_pickle=True)
_, ax = plt.subplots(figsize=(6, 4), tight_layout=True)
ax.plot(sim_strain_stable, sim_stress_stable, 'r', label='non-stable')
ax.plot(sim_strain_nonstable, sim_stress_nonstable, '--b', label='stable')
ax.set_xlabel('plastic strain $\\varepsilon-\\varepsilon_y$')
ax.set_ylabel('stress $\sigma-\sigma_y$[MPa]')
ax.legend(loc='upper left')
ax.grid(alpha=0.2)
plt.savefig("")
plt.show()

In [ ]:
target = '7min_523K'
rootdir = ''

taskname = 's1ag1-Wstable'
handler = HandlerDamask(yaml, taskname, target, rootdir, solver)
for _ in handler.params_updater():
	ss_stable, pp_stable = handler.getDAMASK3(return_aux=True, plastic=True)
	
taskname = 's1ag1-Wnonstable'
handler = HandlerDamask(yaml, taskname, target, rootdir, solver)
for _ in handler.params_updater():
	ss_nonstable, pp_nonstable = handler.getDAMASK3(return_aux=True, plastic=True)
	
_, ax = plt.subplots(figsize=(6, 4), tight_layout=True)
ax.plot(sim_strain_stable[:-1], ss_stable, linestyle='--', color='darkblue', label='stable, $\sigma_s$')
ax.plot(sim_strain_stable[:-1], pp_stable, linestyle='--', color='darkred', label='stable, $\sigma_p$')
ax.plot(sim_strain_nonstable[:-1], ss_nonstable, color='darkblue', label='nonstable, $\sigma_s$')
ax.plot(sim_strain_nonstable[:-1], pp_nonstable, color='darkred', label='nonstable, $\sigma_p$')

ax.set_xlabel('plastic strain $\\varepsilon-\\varepsilon_y$')
ax.set_ylabel('stress')
ax.legend()
ax.grid(alpha=0.2)
plt.savefig("")
plt.show()